## Data Cleaning

In [8]:
import pandas as pd

df = pd.read_csv(r"C:\Users\Ruth\Downloads\CT4101 Portfolio\Milestone 1\applications.csv",low_memory=False)

df_model = df.copy()

In [9]:
df_model = df_model.drop_duplicates()
print(df_model.shape)

(42862, 64)


Removing duplicate rows

In [13]:
df_model["submission_date"] = pd.to_datetime(
    df_model["submission_date"], errors="coerce"
)
df_model["date_of_decision"] = pd.to_datetime(
    df_model["date_of_decision"], errors="coerce"
)
df_model["processing_time_days"] = ( 
    df_model["date_of_decision"] - df_model["submission_date"]
).dt.days

In [16]:
print("Missing processing times:", 
      df_model["processing_time_days"].isna().sum())
print("Negative processing times:", 
     (df_model["processing_time_days"] < 0).sum())
      

Missing processing times: 1
Negative processing times: 2


In [17]:
df_model = df_model.dropna(subset=["processing_time_days"])
df_model = df_model[df_model["processing_time_days"] >=0].copy()
print(df_model.shape)

(42859, 65)


757 exact duplicate rows were removed. The dataset now has 65 columns as processing_time_days was added. The number of rows reduced because there was invalid processing time values. The reaming non-negative and non-missing values can be used for modelling.

In [19]:
print("Unique applications:", 
      df_model["application_reference_no"].nunique())
print("Repeated application references:",
      df_model["application_reference_no"].duplicated().sum())

Unique applications: 42711
Repeated application references: 148


In [20]:
processing_times = df_model.groupby(
    "application_reference_no"
)["processing_time_days"].nunique()
print("Application with different processing times:",
      (processing_times > 1).sum())

Application with different processing times: 0


None of the repaeted application references have different processing times. 

## Feature Selection

In [29]:
candidate_features = [
    "application_type", "local_authority",  "classification_of_works_or_building", "sub_group", 
    "use_of_proposed_works_or_building", "main_construction_type", "site_area", "floor_area_of_building", "total_area_of_ground_floor", 
    "no_of_stories_above_ground_level", "no_of_basement_stories", "height_of_top_floor_above_ground_level"
]

feature_counts = df_model.groupby(
    "application_reference_no"
)[candidate_features].nunique(dropna=False)

print("Applications with different predictor values:",
      feature_counts.gt(1).any(axis=1).sum())

Applications with different predictor values: 0


Checks whether any repeated application reference has different values for proposed predictors. The repeated application references havde identical processing times and identical values accross all 12 selected predictor columns. 

The submission date was reatined for creating the chronological train/test split but excluded from the initial predictor set. The selected predictors consist of application characteristics and building measurements avalailable at submission. Date-derived features may be considered in later modelling stages. 

Going to retain one row per application, as the goal of this project is predicting processing times for individual building control applications. 

In [30]:
df_model = df_model.drop_duplicates(
    subset=["application_reference_no"]
).copy()
print(df_model.shape)

(42711, 65)


## Train/Test Split

In [32]:
from sklearn.model_selection import train_test_split

df_model = df_model.sort_values("submission_date")

X = df_model[candidate_features]
y = df_model["processing_time_days"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,
    shuffle=False
)

print("Training observations:", len(X_train))
print("Testing observations:", len(X_test))

Training observations: 34168
Testing observations: 8543


A time-based train/test split was selected because the aim is to predict processing times for future building control applications. The dataset was ordered by submission date, ane the earliest 80% of the observations were assigned to training, and the latest 20% reserved for testing. This follows the time-aware principles from week 4 notes.

## Preprocessing Pipeline

In [33]:
print(X_train.dtypes)

application_type                           object
local_authority                            object
classification_of_works_or_building        object
sub_group                                  object
use_of_proposed_works_or_building          object
main_construction_type                     object
site_area                                 float64
floor_area_of_building                    float64
total_area_of_ground_floor                float64
no_of_stories_above_ground_level          float64
no_of_basement_stories                    float64
height_of_top_floor_above_ground_level    float64
dtype: object


Objects are categorical and float64 values are numerical.

In [35]:
categorical_features = X_train.select_dtypes(
    include = ["object"]
).columns.tolist()

numerical_features = X_train.select_dtypes(
    include = ["float64", "int64"]
).columns.tolist()

print("Categorical features:", categorical_features)
print("Numerical features:", numerical_features)
    

Categorical features: ['application_type', 'local_authority', 'classification_of_works_or_building', 'sub_group', 'use_of_proposed_works_or_building', 'main_construction_type']
Numerical features: ['site_area', 'floor_area_of_building', 'total_area_of_ground_floor', 'no_of_stories_above_ground_level', 'no_of_basement_stories', 'height_of_top_floor_above_ground_level']


In [38]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

numerical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

- Pipeline groups processing operations together
- SimpleImputer handles missing numerical values
- strategy="median" replaces missing values with the median of the corresponding numerical column

In [41]:
from sklearn.preprocessing import OneHotEncoder

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

- SimpleImputer(strategy="most_frequent") replaces missing categorical values with the most frequent occuring one in each column
- OneHotEncoder converts categorical values into numerical columns that a machine learning model can use
- handle_unknown="ignore" prevents errors if the test dataset contains a category that wasn't present in the training dataset. 

In [43]:
from sklearn.compose import ColumnTransformer 

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

- ColumnTransformer applies different preprocessing operationd to different columns
- "num" applies numerical pipeline to 6 numerical predictors
- "cat" applies categorical pipline to 6 categorical predictors

## Baseline Model

In [46]:
from sklearn.dummy import DummyRegressor

baseline_model = Pipeline(steps=[
    ("preprocessor", preprocessor), 
    ("regressor", DummyRegressor(strategy="median"))
])

- DummyRegressor creates a simple baseline model against which more advanced models can be compared
- strategy="median" means the model predicts the median processing time from the training data for every application, median is used because processing time distribution contains some unusually long durations, so it provides an appropraite baseline. It's also appropraite when evaluating predictors using MAE
- Pipeline combines preprocessing and the baseline model into one workflow

In [47]:
baseline_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['site_area',
                                                   'floor_area_of_building',
                                                   'total_area_of_ground_floor',
                                                   'no_of_stories_above_ground_level',
                                                   'no_of_basement_stories',
                                                   'height_of_top_floor_above_ground_level']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['application_type',
                                                   'local_authority',
                                                   'classification_of_works_or_building',
                                                   'sub_group',
                                                   'use_of_proposed_works_or_building',
                                                   'main_construction_type'])])),
                ('regressor', DummyRegressor(strategy='median'))])

- fit() trains the model using the 34,168 training observations
- The preprocessing pipeline learns how to handle missing values and encode categorical variables using the training data only.


In [48]:
y_pred = baseline_model.predict(X_test)

print(y_pred[:10])

[56. 56. 56. 56. 56. 56. 56. 56. 56. 56.]


This generates predictions for the 8,543 test predictions, the first 10 are displayed. 

In [49]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

mae = mean_absolute_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)

print("Baseline MAE:", mae)
print("Baseline RMSE:", rmse)

Baseline MAE: 32.86070467049046
Baseline RMSE: 52.780713323474515


The higher RMSE indictes that some predictions have relatively large errors. These results establish a bseline against